# Strands Lab: Agents, Tools, and a Goal-Driven Worker

This reference builds from a simple OpenAI-backed Strands agent to a worker that reads notes, translates them into Tagalog, writes a file, and records progress on a SQLite todo board.

| Stage | Main concept | Key APIs |
|---|---|---|
| Setup and Steps 1-2 | Configure a model and run an agent asynchronously | `OpenAIModel`, `Agent`, `invoke_async` |
| Todo board | Persist and display goals and child steps | `board.reset_board`, `board.list_todos`, `board.show_board` |
| Step 3 | Expose Python functions as tools | `@tool`, `tools=[...]` |
| Step 4 | Add filesystem tools through MCP | `MCPClient`, `stdio_client`, `StdioServerParameters` |
| Step 5 | Combine planning, file operations, and progress tracking | Worker agent with board and MCP tools |

**Prerequisites:** the project's Python environment, `OPENAI_API_KEY`, the local `board.py` module, Node.js/`npx`, and an existing `workspace` folder containing `notes.txt`. Run from this notebook's project directory so relative workspace paths resolve as intended.

**Before running:** model requests can incur charges; `npx -y` can download the filesystem server; the worker can write or overwrite files in its allowed workspace. Both `board.reset_board()` calls delete existing todos from the configured board. `load_dotenv(override=True)` lets matching `.env` values replace environment settings.

**Review scope:** comments and Markdown only. Saved outputs show a completed example, not a fresh verification of translation quality. No external model requests, MCP server launches, board resets, or file-writing cells were rerun during this review.

In [2]:
import os
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from strands import Agent, tool
from strands.models.openai import OpenAIModel
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

# Matching .env values override existing environment settings.
load_dotenv(override=True)

True

## Step 1: Create The Agent

`OpenAIModel` configures the provider client and selects the model with `model_id`. `Agent(model=model, ...)` then attaches that model to a system prompt. These are different parameters: `model_id` belongs to the provider wrapper; `model` belongs to the agent. The API key must be present before constructing the model.

In [5]:
MODEL = "gpt-5.4-mini"

model = OpenAIModel(client_args={"api_key": os.environ["OPENAI_API_KEY"]}, 
                    model_id=MODEL)

agent = Agent(
    model=model,
    system_prompt="You are a concise, friendly assistant. Reply in a single short sentence.",
)

## Step 2: Run It

`invoke_async` runs the agent without blocking Jupyter's event loop. The default callback displays generated text, as shown in the saved output; `result` stores the returned run result. A reused Strands agent retains conversation history, so rerunning a prompt on the same instance is not necessarily a fresh conversation.

In [6]:
result = await agent.invoke_async("Say hello in Tagalog.")

Kamusta!

# Project: SQLite Todo Board

The board stores goals and child steps as rows. A goal has `parent_id=None`; a step points to its goal's ID. Statuses are `pending`, `in_progress`, and `done`. The default database is beside `board.py`, unless `BOARD_PATH` was configured before importing that module. This board is separate from the agent's conversation history.

In [7]:
import board

# Destructive setup: drop existing todos and create an empty table.
board.reset_board()
board.add_goal("Read notes.txt, translate its contents into Tagalog and write the Tagalog to tagalog.txt")
board.list_todos()

[{'id': 1,
  'parent_id': None,
  'task': 'Read notes.txt, translate its contents into Tagalog and write the Tagalog to tagalog.txt',
  'status': 'pending',
  'result': ''}]

In [8]:
# Human-readable Rich display; this does not change board rows.
board.show_board()

Goal #1: Read notes.txt, translate its contents into Tagalog and write the Tagalog to tagalog.txt

## Step 3: Add Tools

`@tool` exposes each function to Strands. The function name, annotations, and docstring describe the tool and its arguments to the model. `show_todos` reads rows, `plan_steps` inserts children, and `complete_task` records completion. The functions perform database work only when called, not when defined.

The introductory `board_agent` registers only listing and completion tools, which is enough for its status question. The final worker also registers planning. Although this example asks only for status, the introductory agent is not read-only because it has `complete_task`.

In [9]:
# Return structured rows to the model, rather than the formatted board display.
@tool
def show_todos() ->list[dict]:
    """ List every todo item on the board. A goal has parent_id None, a step has parent_id set to its goal's id."""
    return board.list_todos()

@tool
def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """ Break a goal int an ordered checklist of steps on the board.
    
    Args:
        goal_id: The id of the goal to break down.
        steps: Short descriptions of the steps to take, in order.
        """
    return {
        'goal_id': goal_id,
        # This is a set comprehension: returned IDs have no ordering guarantee.
        'step_ids': {board.add_step(goal_id, step)for step in steps}
    }

@tool
def complete_task(task_id: int, result: str) -> dict:
    """ Mark a todo (a step or the goal) done and record a short result summary.
    
    Args:
        task_id: The id of the todo to mark done.
        result: A short summary of what was accomplished.
        """
    # The helper updates the row, but does not check task existence or prerequisites.
    board.complete_todo(task_id, result)
    return{
        'task_id': task_id,
        'status': 'done'
    }



In [11]:
board_agent = Agent(
    model=model,
    system_prompt="You help manage a shared todo board.",
    tools=[
        show_todos,
        complete_task]
)

In [12]:
result = await board_agent.invoke_async("What is on the board right now, and what is its status?")


Tool #1: show_todos
There is 1 item on the board:

1. **Read notes.txt, translate its contents into Tagalog and write the Tagalog to tagalog.txt** — **pending**

## Step 4: Add MCP

`MCPClient` wraps a factory that opens the filesystem server over standard input/output. The resolved workspace is supplied as the server's allowed root and subprocess working directory. Passing the MCP client in `tools` makes its file operations available to the agent. The server enforces its file-tool root; this does not restrict Python code or the separate SQLite board to that directory.

`startup_timeout=60` bounds server startup, not the entire agent run. The saved read-only example shows a `read_text_file` call followed by a summary.

In [13]:
# resolve() uses the kernel's working directory; it does not create the folder.
workspace = Path('workspace').resolve() # The only folder the agent may touch.

filesystem = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", 
                  str(workspace)],
            cwd=str(workspace)
        )
    ),
    startup_timeout=60
)

In [14]:
file_agent = Agent(
    model=model,
    system_prompt="You can read and write files in your workspace. Use your tools to do what is asked.",
    tools=[filesystem]
)

In [15]:
result = await file_agent.invoke_async("Read notes.txt and summarize it in one short sentence.")



Tool #1: read_text_file
The notes say the team is building a small language tutor one task at a time, with each helper carefully completing and marking off a shared task.

## Step 5: Put It In A Loop With A Goal

The worker combines board operations with file tools. Its SDK-managed loop calls the model, executes requested tools, feeds results back, and continues toward a final answer; there is no explicit Python `for` loop here.

The cell replaces the board, creates and claims a goal, then asks the worker to plan and execute it. The saved output shows tool calls for listing, planning, reading, writing, and four completions, with the goal and three steps recorded as done. Those records reflect tool updates; they are not independent checks of the written translation.

In [16]:
INSTRUCTIONS =""" 
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan;
the handful of concrete steps the work itself breaks down into, added to the
board under the goal. Then carry them out with your file tools, marking each
step done as you finish it. Once the steps are all done, close the goa.. Your
files live in the single folder your tools are allowed to use.
"""

worker =Agent(
    model=model,
    system_prompt=INSTRUCTIONS,
    tools=[
        show_todos,
        plan_steps,
        complete_task,
        filesystem
    ]
)

# Resetting here discards the board created in the earlier demonstration.
board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contents into Tagalog, and write the Tagalog to tagalog.txt.")
# The goal is now in_progress, despite the pending-goal wording below.
board.claim_todo(goal_id)

# The default callback prints the answer; this invocation's result is not assigned.
await worker.invoke_async("Please work the pending goal on the board.")
board.show_board()


Tool #1: show_todos

Tool #2: plan_steps

Tool #3: read_text_file

Tool #4: write_file

Tool #5: complete_task

Tool #6: complete_task

Tool #7: complete_task

Tool #8: complete_task
Done — I translated `notes.txt` into Tagalog and wrote it to `tagalog.txt`.

Goal #1: Read notes.txt, translate its contents into Tagalog, and write the Tagalog to tagalog.txt.  Completed the translation task and saved the Tagalog version to tagalog.txt.
  Step #2: Read notes.txt and understand the source text  Read notes.txt and extracted the source text for translation.
  Step #3: Translate the notes into Tagalog  Translated the notes into Tagalog.
  Step #4: Write the Tagalog translation to tagalog.txt  Wrote the Tagalog translation to tagalog.txt.

## Conclusion and Quick Reference

The notebook demonstrates a coherent progression: configure a model, invoke an agent, define board tools, connect filesystem tools, and combine them into a task-oriented worker. `model_id` is correctly used in the current `OpenAIModel` setup.

- **Model and prompt:** `OpenAIModel` selects the backend; each agent's `system_prompt` describes its role.
- **Tools and loop:** `@tool` exposes Python operations; `MCPClient` supplies external operations. The agent chooses tool calls during its run.
- **State:** SQLite stores task progress; the agent separately maintains conversation history. Resetting the board does not clear an already-created agent's history.
- **Output:** default callbacks display text and tool activity. A returned run result and the stored board rows are different artifacts.

**Validation:** the saved outputs support the demonstrated happy path. An isolated test of the undecorated `plan_steps` function with a fake board confirmed that insertion calls follow the supplied step sequence, but `step_ids` is a set and fails standard `json.dumps` serialization. This does not imply the saved Strands run failed; its output shows that it completed. Notebook structure, Python syntax, and unchanged executable code are checked separately without rerunning the workflow.

## Suggested Logic Improvements - Not Applied

All executable code, prompts, and tool docstrings remain unchanged. The suggestions below describe their implications before any future implementation.

| Suggestion | Current behavior and implication of the change |
|---|---|
| Return a list of IDs from `plan_steps` | Replace the set comprehension with a list comprehension. Inserts already follow the input sequence, but a list would preserve that order in the returned result and be standard-JSON serializable. This changes the tool result's value type from set to list. |
| Address the claimed goal explicitly | `claim_todo` makes the goal `in_progress`, while both worker prompts refer to a pending goal. Passing the claimed goal ID and describing it as in progress would remove ambiguity and narrow task selection. |
| Validate task IDs and completion prerequisites | `add_step` does not verify its parent exists, and `complete_todo` can update zero rows without an error. The tool still returns `status='done'`. Checking existence and confirming all children are done before closing a goal would reject invalid or premature completions instead of reporting success. |
| Independently verify the output file | After writing, check the destination file exists and has nonempty content before marking the goal done. This verifies a basic artifact, not translation quality; accurate translation still needs an appropriate review. |
| Make resets intentional and isolate demonstration data | Both resets drop all todos in the configured board. A dedicated demo database or explicit reset confirmation would protect unrelated work, while changing where data is stored or how the notebook is rerun. |
| Check prerequisites before starting | Check the API key, `npx`, workspace directory, and source file before paid calls or destructive resets. Resolve the intended project path explicitly or require the documented working directory. Fail clearly rather than silently selecting a different folder. |
| Correct prompt and tool-description typos | `close the goa..` and `Break a goal int` are unclear. Fixing them changes text the model sees, even though no Python control flow changes. |
| Use least-privilege tools where appropriate | The status-only `board_agent` also has `complete_task`. Removing it would prevent that agent from modifying the board but would also remove its intentional completion capability. |
| Make reruns and tool delivery reproducible | Recreate agents when a fresh conversation is intended, and pin a tested filesystem-server package version rather than downloading the current version through unversioned `npx -y`. These choices change conversational context and dependency-update behavior. |
| Retain the final worker result if needed | Assign the final `await worker.invoke_async(...)` to a separate variable for inspection; the existing `result` still belongs to the earlier file-summary run. This would add a reference to the final result without changing the requested task. |

**Not changed:** unused imports, formatting, tool schemas, model settings, prompts, board helpers, task statuses, and file operations.